In [1]:
import os

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
if os.environ["OPENAI_API_KEY"]:
    print("API key is set")

API key is set


In [3]:
print(os.getenv("OPENAI_API_KEY") is not None)

True


In [4]:
from langchain_openai import ChatOpenAI

In [5]:
llm = ChatOpenAI(model="gpt-5-nano",temperature=0)

In [6]:
response = llm.invoke("Hello, explain RAG in one sentence.")
print(response.content)

RAG (Retrieval-Augmented Generation) is a framework that retrieves relevant passages from an external knowledge source and then generates answers conditioned on those retrieved passages.


In [7]:
response = llm.invoke("Tell me what is AI in one line")
print(response)
print(response.content)

content='AI is the field of computer science focused on creating systems that can learn, reason, and act to solve tasks that usually require human intelligence.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 293, 'prompt_tokens': 14, 'total_tokens': 307, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ELEISOXmermCZtQ3ylEK0zS057nf7', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a07887-b6cd-7de0-bcb2-88a381d4ae6f-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 14, 'output_tokens': 293, 'total_tokens': 307, 'input_token_detail

# RAG IMPLEMENTATION WITH YOUR OWN TEXT DATA


### STEP 1: Extracting text from pdf 

In [8]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "./Docs/NISM-8.pdf"

loader = PyPDFLoader(pdf_path)

docs = loader.load()

C:\Users\harsh\AppData\Local\Temp\ipykernel_16088\1939168610.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [9]:
docs[0]

Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Acrobat PDFMaker 9.0 for Word', 'creationdate': '2014-06-03T15:46:09+05:30', 'author': 'mitu', 'moddate': '2014-06-03T15:46:09+05:30', 'source': './Docs/NISM-8.pdf', 'total_pages': 162, 'page': 0, 'page_label': '1'}, page_content='')

In [10]:
docs

[Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Acrobat PDFMaker 9.0 for Word', 'creationdate': '2014-06-03T15:46:09+05:30', 'author': 'mitu', 'moddate': '2014-06-03T15:46:09+05:30', 'source': './Docs/NISM-8.pdf', 'total_pages': 162, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Acrobat PDFMaker 9.0 for Word', 'creationdate': '2014-06-03T15:46:09+05:30', 'author': 'mitu', 'moddate': '2014-06-03T15:46:09+05:30', 'source': './Docs/NISM-8.pdf', 'total_pages': 162, 'page': 1, 'page_label': '2'}, page_content='1 \n \nWorkbook for \nNISM-Series-VIII:  \nEquity Derivatives \nCertification Examination \n \n \n \n \n \n \n \n \n \nNational Institute of Securities Markets  \nwww.nism.ac.in'),
 Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Acrobat PDFMaker 9.0 for Word', 'creationdate': '2014-06-03T15:46:09+05:30', 'author': 'mitu', 'moddate': '2014-06-03T15:46:09+05:30', 'source': '

In [11]:
len(docs)

162

#### Creating own Metadata for PDF Chunks

In [12]:
docs[0]

Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Acrobat PDFMaker 9.0 for Word', 'creationdate': '2014-06-03T15:46:09+05:30', 'author': 'mitu', 'moddate': '2014-06-03T15:46:09+05:30', 'source': './Docs/NISM-8.pdf', 'total_pages': 162, 'page': 0, 'page_label': '1'}, page_content='')

In [13]:
docs[0].metadata

{'producer': 'Adobe PDF Library 9.0',
 'creator': 'Acrobat PDFMaker 9.0 for Word',
 'creationdate': '2014-06-03T15:46:09+05:30',
 'author': 'mitu',
 'moddate': '2014-06-03T15:46:09+05:30',
 'source': './Docs/NISM-8.pdf',
 'total_pages': 162,
 'page': 0,
 'page_label': '1'}

In [15]:
for i in docs:
    i.metadata = {"source": "NISM-8.pdf",
                  "document_id": "8"}

In [16]:
docs[0].metadata

{'source': 'NISM-8.pdf', 'document_id': '8'}

### STEP 2: Splitting the Document into CHUNKS

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [18]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100)

chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'NISM-8.pdf', 'document_id': '8'}, page_content='1 \n \nWorkbook for \nNISM-Series-VIII:  \nEquity Derivatives \nCertification Examination \n \n \n \n \n \n \n \n \n \nNational Institute of Securities Markets  \nwww.nism.ac.in'),
 Document(metadata={'source': 'NISM-8.pdf', 'document_id': '8'}, page_content='2 \n \nThis workbook has been developed to assist candidates in preparing for the National \nInstitute of Securities Markets (NISM) NISM- Series-VIII: Equity Derivatives Certification \nExamination (NISM-Series-VIII: ED Examination). \n \nWorkbook Version:  April 2014 \n \nPublished by: \nNational Institute of Securities Markets  \n© National Institute of Securities Markets, 2012 \nPlot 82, Sector 17, Vashi \nNavi Mumbai – 400 703, India \n \nAll rights reserved. Reproduction of this publication in any form without prior \npermission of the publishers is strictly prohibited.'),
 Document(metadata={'source': 'NISM-8.pdf', 'document_id': '8'}, page_conten

In [19]:
len(chunks)

440

### STEP 3: Creating Embeddings for the Chunks

In [20]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")  ## model="text-embedding-3-small" is default

In [21]:
embedding_model.embed_query("What is nism?")

[-0.0277862548828125,
 0.038421630859375,
 -0.004810333251953125,
 0.00106048583984375,
 0.0133056640625,
 0.01395416259765625,
 -0.0032863616943359375,
 0.03277587890625,
 0.0023899078369140625,
 -0.025787353515625,
 0.012420654296875,
 -0.016326904296875,
 -0.073486328125,
 -0.03546142578125,
 0.0731201171875,
 0.0205230712890625,
 -0.056854248046875,
 0.0079345703125,
 0.0036334991455078125,
 0.0249786376953125,
 0.050811767578125,
 0.034332275390625,
 -0.041473388671875,
 0.0190887451171875,
 0.00197601318359375,
 -0.0164642333984375,
 -0.0179290771484375,
 -0.00897216796875,
 0.07952880859375,
 -0.0007672309875488281,
 0.072509765625,
 -0.03814697265625,
 0.01319122314453125,
 -0.0130767822265625,
 -0.002872467041015625,
 0.0115966796875,
 -0.001922607421875,
 -0.01513671875,
 -0.004638671875,
 -0.0028743743896484375,
 -0.048980712890625,
 -0.041656494140625,
 -0.0189361572265625,
 0.0303497314453125,
 -0.0279388427734375,
 0.007732391357421875,
 -0.0194244384765625,
 -0.013000488

In [22]:
vector = embedding_model.embed_query("What is nism?")
print(len(vector))


1536


In [39]:
vector = embedding_model.embed_query("What is Research Analyst?")
print(len(vector))


1536


### STEP 4: Create and Store Embeddings in Existing Local Vector Store

In [40]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma(persist_directory="./Vector_store/",
                     embedding_function=embedding_model)

In [41]:
vectorstore.add_documents(chunks)

['d38cb79e-fad9-46de-8123-6bd636adc213',
 'd711fa41-c056-4ffd-a344-97bf58e23edb',
 '7276c036-eb6e-427c-b272-e2509d429214',
 '359822c3-f057-433a-8050-7b9b4dcd443e',
 '60f64c24-a133-444b-af29-ede4fd46d17d',
 'ff9f064f-d7a2-442b-bc85-c8a0c4ee02b8',
 'bb847ed5-79cc-47d0-8bcf-bb0002dcad14',
 '00df1850-5baf-4e95-b84e-02f5364791c1',
 '17fd5ada-f164-4b81-9f7b-18ec18d15fc5',
 '66032168-ecbf-4a5b-8404-12384779f1f0',
 '6c1dc9ca-d8c2-43e3-98e8-abbd2fffb1c9',
 'cbb0c937-df91-4923-b670-3761eea12d6a',
 'c04df32e-eede-4e35-bd02-916223b63eef',
 '4d7be5c2-dff5-4f51-9794-689334af2bda',
 'fc3eca95-835e-47de-abf6-dbb39f998c8e',
 '402e021e-97c5-4954-a58a-3b08a54e10c6',
 '28674f00-83cb-4671-9c00-19e73b9168e9',
 'b8697b43-9ee5-4d13-a8b6-2ba68472bb9d',
 'f36eaebc-0624-46e5-bb65-22ab837f2aa2',
 '9e0d512a-a670-42f8-a436-0e26e0bc2662',
 '2e0c5591-c19f-45d0-8909-b183fb57c71c',
 '0df71e2f-bcc5-4d1f-8f09-737a31b799ef',
 '33873050-366a-4f15-b826-dc06a22ebade',
 '5f9a0a97-f039-42a5-98dc-2d15d963af52',
 'd8f374df-38dc-

In [42]:
import chromadb
print(chromadb.__version__)

1.5.9


In [43]:
import sys
print(sys.executable)

import site
print(site.getsitepackages())


d:\rag-1-project\rag_venv\Scripts\python.exe
['d:\\rag-1-project\\rag_venv', 'd:\\rag-1-project\\rag_venv\\Lib\\site-packages']


In [44]:
# vectors = []
# for doc in chunks:
#     vector = embedding_model.embed_documents([doc.page_content])
#     vectors.append(vector)
# print(vectors)    
# print("Number of vectors:", len(vectors))
# print("Dimensions of first vector:", len(vectors[0]))


### STEP 5: Semantic Search

In [45]:
vectorstore.similarity_search("What is NISM?", k= 3)

[Document(metadata={'document_id': '8', 'source': 'NISM-8.pdf'}, page_content='NISM brings out various publications on securities markets with a view to enhance \nknowledge levels of participants in the securities industry. \n \nNISM is mandated to implement certification examinations for professionals employed \nin various segments of the Indian securities markets.'),
 Document(metadata={'document_id': '8', 'source': 'NISM-8.pdf'}, page_content='4 \n \nAbout NISM  \n \nIn pursuance of the announcement made by the Finance Minister in his Budget Speech \nin February 2005, Se curities and Exchange Board of India (SEBI) has established the \nNational Institute of Securities Markets (NISM) in Mumbai. \n \nSEBI, by establishing NISM, has articulated the desire expressed by the Indian \ngovernment to promote securities market education and research. \n \nTowards accomplishing the desire of Government of India and vision of SEBI, NISM has \nlaunched an effort to deliver financial and securiti

### Talk to LLM

In [46]:
context = vectorstore.similarity_search("What is NISM?", k= 3)

In [47]:
response = llm.invoke(f"What is NISM? You can answer using the following context: {context}")
print(response.content)
print(10*"*")
print(response)

NISM stands for the National Institute of Securities Markets. It is an institution established by the Securities and Exchange Board of India (SEBI) in Mumbai (announced in 2005) to promote education and research in the securities markets.

Key points:
- Purpose: Promote securities market education and research; publish materials to enhance knowledge; implement certification examinations for professionals across India’s securities markets.
- Structure: Six schools serving investors, issuers, intermediaries, regulatory staff, policymakers, academia, and future professionals.
- Offerings: Certification examinations, training programs, structured learning plans, and supportive workbooks; benchmarks for knowledge across product areas like equities, mutual funds, derivatives, compliance, operations, advisory, and research.
**********
content='NISM stands for the National Institute of Securities Markets. It is an institution established by the Securities and Exchange Board of India (SEBI) in 

In [48]:
response = llm.invoke(f"What is agentic coding? You can answer using the following context: {context}")
print(response.content)
print(5*"*")
print(response)

The term "agentic coding" does not appear in the provided materials. The excerpts you shared discuss NISM’s role, its establishment by SEBI, and the certification programs for securities market professionals, but there is no definition or discussion of "agentic coding."

If you have a different source or can share more context (e.g., a field like psychology, AI, or cognitive science), I can help interpret or define it. Would you like me to look for definitions in other documents or clarify what context you have in mind?
*****
content='The term "agentic coding" does not appear in the provided materials. The excerpts you shared discuss NISM’s role, its establishment by SEBI, and the certification programs for securities market professionals, but there is no definition or discussion of "agentic coding."\n\nIf you have a different source or can share more context (e.g., a field like psychology, AI, or cognitive science), I can help interpret or define it. Would you like me to look for defi

### Re-Use the Vector Database

In [49]:
vectorstore_persist = Chroma(
    persist_directory="./Vector_store/",
    embedding_function=embedding_model
)

In [50]:
vectorstore_persist.similarity_search("Why is primary role of RA?", k= 3)

[Document(metadata={'source': 'NISM-15.pdf', 'document_id': '15'}, page_content='21 \n \nSample Questions \n1. What is the role of Research Analyst? \na. RAs are only involved in the analysis of data \nb. RAs are only involved in collection of the data \nc. RAs help their clients take informed decisions \nd. RAs help in financial planning of their client'),
 Document(metadata={'source': 'NISM-15.pdf', 'document_id': '15'}, page_content='• Communication, done through written research reports, should be simple, clear and concise. \n• If there is any conflict of interest (e.g., RA holds shares of the subject company), such information \nshould be disclosed beforehand. \n• Assumptions, if any, must be clearly stated in the research reports. \n• Abbreviations/Jargons should either be avoided or explained clearly in simple words. \nThe role of RA s is to collect data/information from different reliable sources, interpret the \ndata/information and convert it into recommendations that their c

In [51]:
vectorstore_persist.similarity_search("Why is derivatives?", k= 3)

[Document(metadata={'document_id': '8', 'source': 'NISM-8.pdf'}, page_content='• Derivatives market helps in transfer of various risks from those who are exposed \nto risk but have low risk appetite to participants with high risk appetite. For'),
 Document(metadata={'document_id': '8', 'source': 'NISM-8.pdf'}, page_content='9 \n \nChapter 1: Basics of Derivatives \n1.1 Basics of Derivatives \nDerivative is a contract or a product whose value is derived from value of some other \nasset known as underlying. Derivatives are based on wide range of underlying assets. \nThese include: \n• Metals such as Gold, Silver, Aluminium, Copper, Zinc, Nickel, Tin, Lead etc. \n• Energy resources such as Oil (crude oil, products, cracks) , Coal, Electricity , \nNatural Gas etc. \n• Agri commodities such as wheat, Sugar, Coffee, Cotton, Pulses etc, and  \n• Financial assets such as Shares, Bonds and Foreign Exchange. \n \n1.2 Derivatives Market – History & Evolution \nHistory of Derivatives may be mapped

In [52]:
vectorstore_persist.similarity_search("Why is NISM?", k= 3)

[Document(metadata={'document_id': '8', 'source': 'NISM-8.pdf'}, page_content='NISM brings out various publications on securities markets with a view to enhance \nknowledge levels of participants in the securities industry. \n \nNISM is mandated to implement certification examinations for professionals employed \nin various segments of the Indian securities markets.'),
 Document(metadata={'source': 'NISM-8.pdf', 'document_id': '8'}, page_content='4 \n \nAbout NISM  \n \nIn pursuance of the announcement made by the Finance Minister in his Budget Speech \nin February 2005, Se curities and Exchange Board of India (SEBI) has established the \nNational Institute of Securities Markets (NISM) in Mumbai. \n \nSEBI, by establishing NISM, has articulated the desire expressed by the Indian \ngovernment to promote securities market education and research. \n \nTowards accomplishing the desire of Government of India and vision of SEBI, NISM has \nlaunched an effort to deliver financial and securiti

In [53]:
context = vectorstore_persist.similarity_search("What is ESG framework for company analysis", k= 3)

In [54]:
response = llm.invoke(f"What is ESG framework for company analysis? You can answer using the following context: {context}")
print(response.content)
print(10*"*")
print(response)

Here’s a concise overview of the ESG framework for company analysis based on the provided document:

- What ESG stands for: Environment, Social, and Governance. It’s a set of non-financial criteria used to assess a company’s overall performance and potential impact on value.

- What each criterion assesses:
  - Environment: the company’s impact on the environment (e.g., carbon emissions, pollution, resource use).
  - Social: the company’s role in social development (e.g., human rights, gender equality, other social factors).
  - Governance: the quality of the company’s governance practices (e.g., governance standards, transparency).

- Why it’s used: ESG is linked to potential financial advantages, such as reduced regulatory risk for environmentally focused companies, positive recall/value in society aiding recruitment and customer attraction, and lower cost of capital due to strong governance reducing risk perceptions.

- How investors use it:
  - ESG criteria are used to filter or sh